# 🚲 Cyclistic Bike-Share Case Study
## Google Data Analytics Capstone | Case Study 1

---

### Scenario

You are a junior data analyst on the marketing analytics team at **Cyclistic**, a fictional bike-share company operating in Chicago with a fleet of 5,824 geotracked bicycles across 692 docking stations.

Cyclistic's finance team has determined that **annual members are significantly more profitable than casual riders**. The director of marketing, Lily Moreno, believes the company's future growth depends on converting casual riders into annual members — rather than targeting entirely new customers.

Your team needs to answer one central question:

> **"How do annual members and casual riders use Cyclistic bikes differently?"**

The insights from this analysis will inform a new marketing strategy designed to convert casual riders into annual members. The executive team will review and must approve any recommendations — meaning they need to be backed by rigorous data analysis and compelling visualisations.

---

### The Six-Step Analysis Framework

This case study follows the Google Data Analytics process:

| Step | What we do |
|------|-----------|
| **Ask** | Define the business problem and key questions |
| **Prepare** | Understand, source, and assess the data |
| **Process** | Clean and transform the data for analysis |
| **Analyse** | Explore patterns and answer the business question |
| **Share** | Visualise findings for stakeholder communication |
| **Act** | Deliver data-backed recommendations |

---

### 📦 Dataset
| Property | Detail |
|----------|--------|
| **Name** | Cyclistic Trip Data (Divvy Bikes) |
| **Source** | https://divvy-tripdata.s3.amazonaws.com/index.html |
| **Period** | Most recent 12 months of available data |
| **Format** | 12 monthly CSV files (one per month) |
| **Licence** | [Motivate International Inc.](https://www.divvybikes.com/data-license-agreement) |
| **Privacy** | No personally identifiable information — cannot connect rides to individual users |

> ⚠️ **Kaggle Setup:** Search for **"divvy-tripdata"** or **"Cyclistic bike share"** on Kaggle and attach the dataset. Then run the path-finder cell below before Cell 2 to confirm your exact `DATA_PATH`.

### ⚙️ Tools
`pandas` · `numpy` · `plotly` · `seaborn` · `matplotlib` · `glob` · `os`


---
## 1️⃣ ASK — Define the Business Problem

### Business Task

Design marketing strategies to convert casual riders into annual members by understanding how each group uses Cyclistic bikes differently.

### Key Stakeholders
- **Lily Moreno** — Director of Marketing (your manager). Wants data-backed recommendations.
- **Cyclistic Marketing Analytics Team** — Your team. Responsible for collecting, analysing, and reporting.
- **Cyclistic Executive Team** — Detail-oriented. Must approve recommendations before implementation.

### Guiding Questions

Three questions will shape the marketing programme:
1. **How do annual members and casual riders use Cyclistic bikes differently?** ← *This analysis answers this one*
2. Why would casual riders buy a membership?
3. How can Cyclistic use digital media to influence casual riders to become members?

### Success Criteria

A deliverable with:
- A clear statement of the business task ✅
- A description of all data sources used ✅
- Documentation of all cleaning and transformation steps ✅
- A summary of the analysis ✅
- Supporting visualisations and key findings ✅
- Top three recommendations based on the analysis ✅


---
## 2️⃣ PREPARE — Understand and Source the Data


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 │ Setup — Libraries & Style
# ─────────────────────────────────────────────────────────────

import os, glob, warnings
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn           as sns
import plotly.express    as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io     as pio

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)
pio.templates.default = "plotly_white"
os.makedirs("output", exist_ok=True)

PALETTE = {
    "member": "#2563EB",   # blue  — members throughout
    "casual": "#F59E0B",   # amber — casuals throughout
    "neutral": "#64748B",
    "green":  "#059669",
    "red":    "#DC2626",
    "purple": "#7C3AED",
}

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "#F8FAFC",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
    "axes.labelsize":    11,
    "axes.titlesize":    13,
})

print("✅  Libraries loaded.")
print("    Blue = Member  |  Amber = Casual  (consistent throughout all charts)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 │ Path Finder — Run This First to Confirm Your Path
# ─────────────────────────────────────────────────────────────
# This cell scans what Kaggle has mounted and prints every CSV.
# Copy the folder path shown and set DATA_PATH in Cell 3.

for dirname, _, filenames in os.walk('/kaggle/input'):
    csv_files = [f for f in filenames if f.endswith('.csv')]
    if csv_files:
        print(f"📁 {dirname}")
        for f in sorted(csv_files)[:5]:
            print(f"   └── {f}")
        if len(csv_files) > 5:
            print(f"   └── ... and {len(csv_files)-5} more CSV files")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 │ Load & Combine 12 Months of Trip Data
# ─────────────────────────────────────────────────────────────
# ⚠️  Update DATA_PATH to match what Cell 2 printed above.

DATA_PATH = "/kaggle/input/divvy-tripdata/"   # ← update if needed

# Find all CSV files in the folder
csv_files = sorted(glob.glob(DATA_PATH + "*.csv"))
print(f"📂  Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"   {os.path.basename(f)}")

# Load and concatenate into one DataFrame
dfs = []
for f in csv_files:
    temp = pd.read_csv(f, low_memory=False)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print(f"\n✅  Combined dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n── Column Names ──────────────────────────────────────")
for col in df.columns:
    print(f"   {col:<35} {df[col].dtype}")


### Data Assessment — ROCCC Framework

Before trusting any dataset, we assess it against five criteria:

| Criterion | Assessment |
|-----------|-----------|
| **R**eliable | ✅ Data collected directly by Motivate International Inc. from actual bike trips |
| **O**riginal | ✅ First-party data from the bike-share system itself |
| **C**omprehensive | ✅ Covers every single ride — not a sample |
| **C**urrent | ✅ Most recent 12 months |
| **C**ited | ✅ Publicly available under official licence |

**Known limitations:**
- No personally identifiable information — can't track whether individual casual riders are local residents or tourists
- Can't link rides to specific customers across multiple trips
- No pricing data included


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 │ Prepare Phase — Data Audit
# ─────────────────────────────────────────────────────────────

print("── Shape ─────────────────────────────────────────────────")
print(f"  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")

print("\n── Null Value Audit ──────────────────────────────────────")
null_s = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_%":    (df.isnull().sum() / len(df) * 100).round(2),
})
print(null_s[null_s["null_count"] > 0].to_string())

print("\n── Rider Type Breakdown (raw) ────────────────────────────")
print(df["member_casual"].value_counts().to_string())
print(f"  Unique values: {df['member_casual'].unique()}")

print("\n── Bike Type Breakdown ───────────────────────────────────")
print(df["rideable_type"].value_counts().to_string())

print("\n── Sample Rows ───────────────────────────────────────────")
df.head(3)


---
## 3️⃣ PROCESS — Clean and Transform

### Cleaning decisions documented below. Every decision has a reason.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 │ Parse Dates & Engineer Core Features
# ─────────────────────────────────────────────────────────────

print(f"Rows before cleaning: {len(df):,}")

# ── Parse timestamps ──────────────────────────────────────────
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"]   = pd.to_datetime(df["ended_at"],   errors="coerce")

# ── Ride length in minutes ────────────────────────────────────
df["ride_length_min"] = (
    (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
)

# ── Time features ─────────────────────────────────────────────
df["hour_of_day"]  = df["started_at"].dt.hour
df["day_of_week"]  = df["started_at"].dt.day_name()
df["month"]        = df["started_at"].dt.month_name()
df["month_num"]    = df["started_at"].dt.month
df["season"] = df["month_num"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring", 5:"Spring",
    6:"Summer",  7:"Summer", 8:"Summer",
    9:"Autumn", 10:"Autumn", 11:"Autumn",
})

print("✅  Dates parsed and features engineered.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 │ Apply Cleaning Rules (Documented)
# ─────────────────────────────────────────────────────────────

# Rule 1: Remove rows where ride_length ≤ 0
# Reason: A trip that ends before or at the same time it starts
#         is physically impossible — test rides or system errors.
mask_duration = df["ride_length_min"] > 0
print(f"  Rule 1 (ride_length > 0):     removes {(~mask_duration).sum():,} rows")

# Rule 2: Remove rides < 1 minute
# Reason: Sub-1-minute rides are almost always false starts
#         or bikes immediately re-docked after undocking.
mask_min = df["ride_length_min"] >= 1
print(f"  Rule 2 (ride_length >= 1min): removes {(~mask_min & mask_duration).sum():,} rows")

# Rule 3: Remove rides > 24 hours
# Reason: Bikes not returned within 24 hours are considered lost/stolen
#         per Cyclistic policy — not valid leisure or commute trips.
mask_max = df["ride_length_min"] <= 1440
print(f"  Rule 3 (ride_length <= 24hr): removes {(~mask_max).sum():,} rows")

# Rule 4: Remove rows with null started_at or ended_at
mask_dates = df["started_at"].notna() & df["ended_at"].notna()
print(f"  Rule 4 (no null timestamps):  removes {(~mask_dates).sum():,} rows")

# Apply all rules
df_clean = df[mask_duration & mask_min & mask_max & mask_dates].copy()

print(f"\n  Before: {len(df):,}  →  After: {len(df_clean):,}")
print(f"  Removed: {len(df) - len(df_clean):,} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")
print(f"\n✅  Clean dataset ready.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 │ Verify Clean Data
# ─────────────────────────────────────────────────────────────

print("── Rider Type Split (clean) ──────────────────────────────")
counts = df_clean["member_casual"].value_counts()
pcts   = df_clean["member_casual"].value_counts(normalize=True) * 100
for rt in counts.index:
    print(f"   {rt:<10}  {counts[rt]:>10,}  ({pcts[rt]:.1f}%)")

print(f"\n── Ride Length Summary (minutes) ─────────────────────────")
print(df_clean.groupby("member_casual")["ride_length_min"]
              .describe()
              .round(2)
              .to_string())

print(f"\n── Date Range ────────────────────────────────────────────")
print(f"   From: {df_clean['started_at'].min()}")
print(f"   To:   {df_clean['started_at'].max()}")


---
## 4️⃣ ANALYSE — Explore Usage Patterns

We now systematically compare members vs. casual riders across five dimensions:

1. **Ride volume** — who rides more?
2. **Ride duration** — how long does each group ride?
3. **Time patterns** — when does each group ride (hour, day, month, season)?
4. **Bike type preference** — what type of bike does each group choose?
5. **Station patterns** — where does each group start and end their trips?


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 │ Ride Volume & Duration Summary Tables
# ─────────────────────────────────────────────────────────────

summary = df_clean.groupby("member_casual").agg(
    total_rides        = ("ride_id",        "count"),
    mean_duration_min  = ("ride_length_min","mean"),
    median_duration_min= ("ride_length_min","median"),
    max_duration_min   = ("ride_length_min","max"),
    std_duration_min   = ("ride_length_min","std"),
).round(2)
summary["ride_share_%"] = (
    summary["total_rides"] / summary["total_rides"].sum() * 100
).round(1)

print("📊  Overall Summary by Rider Type:")
print(summary.to_string())

print("\n📊  Monthly Ride Counts:")
monthly = (
    df_clean.groupby(["month_num","month","member_casual"])["ride_id"]
            .count()
            .reset_index()
            .rename(columns={"ride_id":"rides"})
            .sort_values("month_num")
)
print(monthly.pivot(index="month", columns="member_casual", values="rides")
             .fillna(0).astype(int).to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9 │ Day of Week & Hour of Day Aggregations
# ─────────────────────────────────────────────────────────────

DOW_ORDER = ["Monday","Tuesday","Wednesday",
             "Thursday","Friday","Saturday","Sunday"]

dow = (
    df_clean.groupby(["day_of_week","member_casual"])
            .agg(rides=("ride_id","count"),
                 avg_duration=("ride_length_min","mean"))
            .reset_index()
)
dow["day_of_week"] = pd.Categorical(dow["day_of_week"],
                                    categories=DOW_ORDER, ordered=True)
dow = dow.sort_values("day_of_week")

hourly = (
    df_clean.groupby(["hour_of_day","member_casual"])
            .agg(rides=("ride_id","count"))
            .reset_index()
)

print("📊  Rides by Day of Week:")
print(dow.pivot(index="day_of_week", columns="member_casual", values="rides")
         .to_string())


---
## 5️⃣ SHARE — Visualisations

Every chart below is designed to answer one specific question. We use consistent colours throughout:
- 🔵 **Blue = Annual Members**
- 🟡 **Amber = Casual Riders**


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10 │ Chart 1 — Total Rides & Avg Duration by Rider Type
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("white")

ride_counts = df_clean["member_casual"].value_counts()
colors_bar  = [PALETTE["member"] if x == "member" else PALETTE["casual"]
               for x in ride_counts.index]

# Left — ride count
bars = axes[0].bar(ride_counts.index, ride_counts.values,
                   color=colors_bar, edgecolor="none", width=0.5)
axes[0].set_title("Total Rides by Rider Type", fontweight="bold")
axes[0].set_ylabel("Number of Rides")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x/1e6):.1f}M" if x >= 1e6 else f"{int(x/1e3)}K"))
axes[0].bar_label(bars, fmt=lambda x: f"{int(x):,}", padding=5, fontsize=10)
axes[0].set_facecolor("#F8FAFC")

# Right — avg ride duration
avg_dur = df_clean.groupby("member_casual")["ride_length_min"].mean()
colors_dur = [PALETTE["member"] if x == "member" else PALETTE["casual"]
              for x in avg_dur.index]
bars2 = axes[1].bar(avg_dur.index, avg_dur.values,
                    color=colors_dur, edgecolor="none", width=0.5)
axes[1].set_title("Average Ride Duration by Rider Type", fontweight="bold")
axes[1].set_ylabel("Average Duration (minutes)")
axes[1].bar_label(bars2, fmt="%.1f min", padding=5, fontsize=10)
axes[1].set_facecolor("#F8FAFC")

plt.suptitle("Ride Volume vs. Duration — Members vs. Casual Riders",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("output/01_volume_and_duration.png", dpi=150, bbox_inches="tight")
plt.show()

print("📌 Key insight: Casual riders take fewer trips but ride significantly longer per trip.")
print("   This suggests members use bikes for routine commutes while casuals use them for leisure.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11 │ Chart 2 — Rides by Day of Week
# ─────────────────────────────────────────────────────────────

fig = px.bar(
    dow, x="day_of_week", y="rides",
    color="member_casual",
    barmode="group",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    title="🗓️  Ride Count by Day of Week — Members vs. Casual Riders",
    labels={"day_of_week": "Day of Week",
            "rides":       "Number of Rides",
            "member_casual": "Rider Type"},
    text_auto=False,
)
fig.update_layout(
    height=440,
    legend_title_text="Rider Type",
    xaxis=dict(categoryorder="array", categoryarray=DOW_ORDER),
    paper_bgcolor="white", plot_bgcolor="#F8FAFC",
)
fig.show()

# Avg duration by day
fig2 = px.line(
    dow, x="day_of_week", y="avg_duration",
    color="member_casual",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    markers=True,
    title="⏱️  Average Ride Duration by Day of Week",
    labels={"day_of_week":  "Day of Week",
            "avg_duration": "Avg Duration (min)",
            "member_casual":"Rider Type"},
)
fig2.update_layout(
    height=400,
    xaxis=dict(categoryorder="array", categoryarray=DOW_ORDER),
    paper_bgcolor="white", plot_bgcolor="#F8FAFC",
)
fig2.show()

print("📌 Members peak mid-week (Tue–Thu) → commuter pattern.")
print("   Casuals peak on weekends (Sat–Sun) → leisure pattern.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12 │ Chart 3 — Rides by Hour of Day
# ─────────────────────────────────────────────────────────────

fig = px.line(
    hourly, x="hour_of_day", y="rides",
    color="member_casual",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    markers=True,
    title="🕐  Ride Count by Hour of Day — Members vs. Casual Riders",
    labels={"hour_of_day":   "Hour of Day (24h)",
            "rides":         "Number of Rides",
            "member_casual": "Rider Type"},
)
fig.add_vrect(x0=7, x1=9, fillcolor="#2563EB", opacity=0.07,
              annotation_text="Morning commute", annotation_position="top left",
              annotation_font_size=9)
fig.add_vrect(x0=16, x1=18, fillcolor="#2563EB", opacity=0.07,
              annotation_text="Evening commute", annotation_position="top right",
              annotation_font_size=9)
fig.update_layout(
    height=430,
    xaxis=dict(tickmode="array",
               tickvals=list(range(0, 24, 2)),
               ticktext=[f"{h:02d}:00" for h in range(0, 24, 2)]),
    paper_bgcolor="white", plot_bgcolor="#F8FAFC",
)
fig.show()

print("📌 Members show two sharp peaks at 8am and 5pm → classic commuter signature.")
print("   Casuals show a gradual single peak in the afternoon → leisure/tourist signature.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 13 │ Chart 4 — Monthly & Seasonal Ride Patterns
# ─────────────────────────────────────────────────────────────

MONTH_ORDER = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

monthly["month"] = pd.Categorical(monthly["month"],
                                   categories=MONTH_ORDER, ordered=True)
monthly = monthly.sort_values("month")

fig = px.line(
    monthly, x="month", y="rides",
    color="member_casual",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    markers=True,
    title="📅  Ride Count by Month — Members vs. Casual Riders",
    labels={"month": "Month", "rides": "Number of Rides",
            "member_casual": "Rider Type"},
)
fig.update_layout(
    height=420,
    xaxis_tickangle=30,
    paper_bgcolor="white", plot_bgcolor="#F8FAFC",
)
fig.show()

# Season heatmap
season_order = ["Spring","Summer","Autumn","Winter"]
season_df = (
    df_clean.groupby(["season","member_casual"])["ride_id"]
            .count()
            .reset_index()
            .rename(columns={"ride_id":"rides"})
)
season_pivot = season_df.pivot(index="season",
                                columns="member_casual",
                                values="rides").reindex(season_order)

fig2, ax = plt.subplots(figsize=(8, 4))
fig2.patch.set_facecolor("white")
sns.heatmap(
    season_pivot, annot=True, fmt=",",
    cmap="YlOrRd", linewidths=0.5,
    linecolor="#E2E8F0",
    annot_kws={"size": 11},
    ax=ax,
    cbar_kws={"label": "Rides", "shrink": 0.7},
)
ax.set_title("Seasonal Ride Distribution — Members vs. Casuals",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("output/02_seasonal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("📌 Both groups peak in summer, but the casual drop-off in winter is much steeper.")
print("   This supports the leisure vs. commute hypothesis — commuters ride year-round.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 14 │ Chart 5 — Bike Type Preference
# ─────────────────────────────────────────────────────────────

bike = (
    df_clean.groupby(["rideable_type","member_casual"])["ride_id"]
            .count()
            .reset_index()
            .rename(columns={"ride_id":"rides"})
)
bike["rideable_type"] = bike["rideable_type"].str.replace("_bike","").str.title()

fig = px.bar(
    bike, x="rideable_type", y="rides",
    color="member_casual",
    barmode="group",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    title="🚲  Bike Type Preference — Members vs. Casual Riders",
    labels={"rideable_type":  "Bike Type",
            "rides":          "Number of Rides",
            "member_casual":  "Rider Type"},
    text_auto=False,
)
fig.update_layout(
    height=420,
    paper_bgcolor="white",
    plot_bgcolor="#F8FAFC",
)
fig.show()

print("📌 Note: Only casual riders use docked bikes.")
print("   Electric bikes are proportionally more popular with casual riders.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 15 │ Chart 6 — Ride Duration Distribution (Box Plot)
# ─────────────────────────────────────────────────────────────

# Cap at 90th percentile for readability
cap = df_clean["ride_length_min"].quantile(0.90)
df_cap = df_clean[df_clean["ride_length_min"] <= cap]

fig = px.box(
    df_cap,
    x="member_casual", y="ride_length_min",
    color="member_casual",
    color_discrete_map={"member": PALETTE["member"],
                        "casual": PALETTE["casual"]},
    points=False,
    notched=True,
    title="📦  Ride Duration Distribution (capped at 90th percentile)",
    labels={"member_casual":   "Rider Type",
            "ride_length_min": "Ride Duration (minutes)"},
)
fig.update_layout(
    height=420,
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="#F8FAFC",
)
fig.show()

print("📌 The notch shows the median confidence interval.")
print("   Non-overlapping notches confirm the median difference is statistically significant.")
print(f"\n   Member median:  {df_clean[df_clean['member_casual']=='member']['ride_length_min'].median():.1f} min")
print(f"   Casual median:  {df_clean[df_clean['member_casual']=='casual']['ride_length_min'].median():.1f} min")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 16 │ Chart 7 — Top 10 Start Stations by Rider Type
# ─────────────────────────────────────────────────────────────

top_stations = {}
for rider in ["member","casual"]:
    top = (
        df_clean[df_clean["member_casual"] == rider]
        .dropna(subset=["start_station_name"])
        .groupby("start_station_name")["ride_id"]
        .count()
        .nlargest(10)
        .reset_index()
        .rename(columns={"ride_id":"rides",
                         "start_station_name":"station"})
    )
    top["rider_type"] = rider
    top_stations[rider] = top

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor("white")

for ax, (rider, data) in zip(axes, top_stations.items()):
    color = PALETTE["member"] if rider == "member" else PALETTE["casual"]
    label = "Annual Members" if rider == "member" else "Casual Riders"
    bars = ax.barh(data["station"], data["rides"],
                   color=color, edgecolor="none", height=0.65)
    ax.set_title(f"Top 10 Start Stations — {label}",
                 fontweight="bold", color=color)
    ax.set_xlabel("Number of Rides")
    ax.invert_yaxis()
    ax.set_facecolor("#F8FAFC")
    ax.bar_label(bars, fmt=lambda x: f"{int(x):,}",
                 padding=4, fontsize=8, color=PALETTE["neutral"])

plt.suptitle("Most Popular Start Stations by Rider Type",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("output/03_top_stations.png", dpi=150, bbox_inches="tight")
plt.show()

print("📌 Casual riders concentrate at tourist/leisure stations (lakefront, parks).")
print("   Members spread across residential and commercial commuter stations.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 17 │ Chart 8 — Executive Summary Dashboard
# ─────────────────────────────────────────────────────────────

member_df = df_clean[df_clean["member_casual"] == "member"]
casual_df = df_clean[df_clean["member_casual"] == "casual"]

total      = len(df_clean)
m_count    = len(member_df)
c_count    = len(casual_df)
m_pct      = m_count / total * 100
c_pct      = c_count / total * 100
m_avg_dur  = member_df["ride_length_min"].mean()
c_avg_dur  = casual_df["ride_length_min"].mean()
m_wknd_pct = member_df[member_df["day_of_week"].isin(["Saturday","Sunday"])].shape[0] / m_count * 100
c_wknd_pct = casual_df[casual_df["day_of_week"].isin(["Saturday","Sunday"])].shape[0] / c_count * 100
m_top_day  = member_df["day_of_week"].value_counts().idxmax()
c_top_day  = casual_df["day_of_week"].value_counts().idxmax()

kpis = [
    ("🔵 Member Rides",        f"{m_count:,}",         f"{m_pct:.1f}% of all rides",  PALETTE["member"]),
    ("🟡 Casual Rides",        f"{c_count:,}",         f"{c_pct:.1f}% of all rides",  PALETTE["casual"]),
    ("⏱️ Member Avg Duration", f"{m_avg_dur:.1f} min",  "per ride",                    PALETTE["member"]),
    ("⏱️ Casual Avg Duration", f"{c_avg_dur:.1f} min",  f"{c_avg_dur/m_avg_dur:.1f}× longer than members", PALETTE["casual"]),
    ("📅 Member Peak Day",     m_top_day,               "highest ride volume",          PALETTE["member"]),
    ("📅 Casual Peak Day",     c_top_day,               "highest ride volume",          PALETTE["casual"]),
    ("🏖️ Member Weekend %",   f"{m_wknd_pct:.1f}%",    "of member rides on weekends",  PALETTE["member"]),
    ("🏖️ Casual Weekend %",   f"{c_wknd_pct:.1f}%",    "of casual rides on weekends",  PALETTE["casual"]),
]

COLS, ROWS = 4, 2
fig = go.Figure()
fig.update_layout(
    width=900, height=380,
    paper_bgcolor="#F8FAFC",
    plot_bgcolor="#F8FAFC",
    margin=dict(l=20, r=20, t=70, b=20),
    title=dict(
        text="📊  Cyclistic — Executive Summary Dashboard",
        font=dict(size=16, color="#1E293B"),
        x=0.5, xanchor="center", y=0.97,
    ),
    xaxis=dict(visible=False, range=[0, COLS]),
    yaxis=dict(visible=False, range=[0, ROWS]),
)

for i, (label, value, sub, color) in enumerate(kpis):
    col = i % COLS
    row = ROWS - 1 - (i // COLS)
    cx  = col + 0.5
    cy  = row + 0.5
    fig.add_shape(type="rect",
        x0=col+0.05, x1=col+0.95,
        y0=row+0.06, y1=row+0.94,
        fillcolor="white", line=dict(color="#E2E8F0", width=1.5), layer="below")
    fig.add_shape(type="rect",
        x0=col+0.05, x1=col+0.105,
        y0=row+0.06, y1=row+0.94,
        fillcolor=color, line=dict(width=0), layer="above")
    fig.add_annotation(x=cx+0.05, y=cy+0.22, text=label,
        showarrow=False, font=dict(size=10, color="#64748B"), xanchor="center")
    fig.add_annotation(x=cx+0.05, y=cy+0.0, text=f"<b>{value}</b>",
        showarrow=False, font=dict(size=16, color=color), xanchor="center")
    fig.add_annotation(x=cx+0.05, y=cy-0.22, text=sub,
        showarrow=False, font=dict(size=9, color="#94A3B8"), xanchor="center")

fig.show()


---
## 6️⃣ ACT — Key Findings & Recommendations

### Summary of Findings

| Finding | Evidence | Implication |
|---------|----------|-------------|
| **Members ride more frequently, casuals ride longer** | Ride count and duration charts | Different motivations: routine transport vs. leisure exploration |
| **Members peak mid-week; casuals peak on weekends** | Day-of-week chart | Members = commuters. Casuals = weekend leisure users |
| **Members show morning and evening commute spikes** | Hour-of-day chart | Members follow a 9-to-5 pattern; casuals show afternoon leisure peaks |
| **Both groups drop off in winter, but casuals drop harder** | Monthly/seasonal chart | Casuals are weather-sensitive. They're not riding out of necessity |
| **Casual riders start at leisure/tourist hotspots** | Station analysis | Casuals may include tourists. Not all are conversion candidates |
| **Casuals use docked bikes; members don't** | Bike type chart | Operational distinction worth noting in membership marketing |

---

### 🎯 Top 3 Recommendations

**Recommendation 1 — Weekend Membership Tier**

Casual riders peak on weekends. Offer a **"Weekend Membership"** at a lower price point than a full annual membership. This meets casuals exactly where their behaviour already is, removing the perceived mismatch of paying for a full membership when they only ride on weekends. As they build the habit, upsell them to full annual membership.

**Recommendation 2 — Seasonal Conversion Campaigns**

Launch targeted digital campaigns in **late Spring (April–May)**, just before casual ridership peaks in summer. This is when casual riders are most active and most likely to consider a membership. Messaging: *"You're already riding — get more value for less."* Include ride history data in the communication (e.g., *"You took 12 rides last summer. An annual membership would have saved you £X."*)

**Recommendation 3 — Station-Level Targeted Activation**

Deploy **physical and digital touchpoints at the top casual rider stations** including the leisure and lakefront hotspots. QR codes at docking stations, push notifications triggered at checkout, and pop-up events during peak summer weekends. These locations are where casual riders already are. Meeting them there with a clear membership pitch is more efficient than broad digital advertising.

---

### What I Would Do With More Time
- **Incorporate pricing data** to calculate actual cost differential between casual passes and membership for the average casual rider, makes the "you'd save £X" message data-backed
- **Tourist vs. local segmentation** — if postal code data were available, separating local casuals from tourists would dramatically sharpen conversion strategy
- **Predictive model** — build a classifier to score which casual riders are most likely to convert based on their usage patterns

---
*Analysis by Jessica Dan-Odhomo · Dataset: Motivate International Inc. (Divvy Bikes) · Tools: Python, Pandas, Plotly, Seaborn*
